# LLM Inference Quantization Benchmark on vLLM

Compares Qwen2.5 in BF16 (FP16 on a T4), AWQ 4-bit and GPTQ 8-bit: throughput, latency and perplexity.

**First:** Runtime → Change runtime type → **T4 GPU** (or A100) → Save.
The vLLM install takes about 5 minutes. The 1.5B run takes about 15 to 20 minutes on a T4.

In [6]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [7]:
!git clone -q https://github.com/riddhixraina/vllm-quant-bench
%cd vllm-quant-bench
!pip install -q -r requirements.txt

/content/vllm-quant-bench/vllm-quant-bench


In [8]:
!grep -c enable_prefix_caching bench_vllm.py

1


## Run all three models

Use `1.5B` on a T4 (16 GB). Use `7B` on an A100.

In [9]:
!./run_all.sh 7B

(MainProcess pid=15099) INFO 10-22 16:42:11 [api_utils.py:286] non-default args: {'max_model_len': 2048, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
(MainProcess pid=15099) INFO 10-22 16:42:12 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=15099) INFO 10-22 16:42:12 [model.py:2152] Using max model len 2048
(MainProcess pid=15099) INFO 10-22 16:42:12 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=15099) INFO 10-22 16:42:12 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files: 100% 10/10 [00:00<00:00, 1753.54it/s]
Download complete:           |  0.00B            
Download complete:           |  0.00B        

In [11]:
!cat results/summary.md

| Run | Quantization | Tok/s @ batch 1 | Tok/s @ batch 4 | Tok/s @ batch 16 | Tok/s @ batch 64 | TTFT (ms) | ms / token | Perplexity |
|---|---|---|---|---|---|---|---|---|
| bf16 | none | 81 | 311 | 987 | 2,148 | 37.0 | 12.04 | 8.123 |
| awq-int4 | auto_awq | 183 (2.24x) | 603 (1.94x) | 1,502 (1.52x) | 2,189 (1.02x) | 42.9 | 5.18 | 8.665 (+6.7%) |
| gptq-int8 | auto_gptq | 130 (1.60x) | 444 (1.43x) | 1,199 (1.21x) | 1,971 (0.92x) | 53.7 | 7.32 | 8.133 (+0.1%) |
GPU: NVIDIA A100-SXM4-40GB. 512-token prompts, 128 output tokens. Ratios and perplexity changes are relative to `bf16`.


To keep the results, download the `results` folder or commit it to the repo.

To test long prompts, run `!OUT=results-2048 ./run_all.sh 7B --input-len 2048 --skip-ppl` and then `!cat results-2048/summary.md`.

In [12]:
!OUT=results-2048 ./run_all.sh 7B --input-len 2048 --skip-ppl
!cat results-2048/summary.md

(MainProcess pid=18582) INFO 10-22 16:50:35 [api_utils.py:286] non-default args: {'max_model_len': 2176, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
(MainProcess pid=18582) INFO 10-22 16:50:36 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=18582) INFO 10-22 16:50:36 [model.py:2152] Using max model len 2176
(MainProcess pid=18582) INFO 10-22 16:50:36 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=18582) INFO 10-22 16:50:36 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files: 100% 10/10 [00:00<00:00, 1796.97it/s]
Download complete:           |  0.00B            
Download complete:           |  0.00B        

In [13]:
!zip -qr results.zip results results-2048
from google.colab import files; files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>